<a href="https://colab.research.google.com/github/saiDan77/nn-from-scratch/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
This task provides a rigorous mathematical derivation of Convolutional Neural Networks (CNNs) using foundational linear algebra and tensor calculus. We will move from defining basic vector spaces to proving translation equivariance, demonstrating how spatial grid structures are processed through sparse, circulant linear operators (convolutions).

## Vector Spaces and Tensors

### Subtask:
Define the mathematical foundations of vector spaces and tensors, focusing on coordinate representations of image data.


### 1. Vector Spaces and Tensors

#### 1.1 Vector Spaces and Basis
A **vector space** $V$ over a field $\mathbb{F}$ (typically $\mathbb{R}$) is a set of objects (vectors) closed under addition and scalar multiplication.
- **Linear Independence**: A set of vectors $\{v_1, ..., v_n\}$ is linearly independent if $\sum a_i v_i = 0 \implies a_i = 0$ for all $i$.
- **Basis**: A set $\mathcal{B} = \{e_1, ..., e_n\}$ is a basis for $V$ if it spans $V$ and is linearly independent. Any vector $x \in V$ can be uniquely represented as $x = \sum x^i e_i$.

#### 1.2 Tensors as Multilinear Maps
An image is not just a vector but is naturally represented as a **tensor**. Mathematically, a tensor of order $k$ is an element of the tensor product of vector spaces $V_1 \otimes V_2 \otimes ... \otimes V_k$.
- For a grayscale image, we consider the space $\mathbb{R}^{H \times W}$, where $H$ and $W$ are spatial dimensions.
- The 'grid' structure is significant because it defines a local topology where the distance between coordinates $(i, j)$ and $(i+1, j)$ corresponds to adjacent physical pixels.

## Linear Transformations and Matrix Properties

### Subtask:
Review linear maps, matrix properties (rank, null space, trace), and the concept of sparsity in linear systems.


### 2. Linear Transformations and Matrix Properties

#### 2.1 Linear Maps
A mapping $L: V \to W$ between vector spaces is a **linear transformation** if for all $u, v \in V$ and $c \in \mathbb{F}$:
1. $L(u + v) = L(u) + L(v)$
2. $L(cu) = cL(u)$

In finite-dimensional spaces, once bases for $V$ and $W$ are fixed, $L$ can be uniquely represented by a matrix $A \in \mathbb{R}^{m \times n}$ such that $L(x) = Ax$.

#### 2.2 Rank, Null Space, and Trace
- **Null Space (Kernel)**: The set $\text{null}(A) = \{x \in V : Ax = 0\}$. It represents the solutions to the homogeneous system.
- **Rank**: The dimension of the image of $A$, denoted $\text{rank}(A)$. By the Rank-Nullity Theorem, $\text{rank}(A) + \dim(\text{null}(A)) = n$.
- **Trace**: For a square matrix, $\text{tr}(A) = \sum_{i=1}^n a_{ii}$, which is the sum of its eigenvalues.

#### 2.3 Sparsity and Efficiency
A matrix is **sparse** if most of its elements are zero. In traditional Fully Connected (FC) layers, the transformation matrix $A$ is dense, requiring $O(N^2)$ parameters for $N$ inputs. For high-resolution images, this is computationally prohibitive. Convolutions address this by imposing a specific sparse structure on $A$, where each output pixel only depends on a small local neighborhood of input pixels.

## The Geometry of Convolution

### Subtask:
Formally derive the 2D discrete convolution from the perspective of a linear operator and define the sliding window mathematically.


### 3. The Geometry of Convolution

#### 3.1 Mathematical Definition of 2D Discrete Convolution
Let $I \in \mathbb{R}^{H \times W}$ be an input image and $K \in \mathbb{R}^{k \times k}$ be a kernel (or filter). The discrete 2D convolution $(I * K)$ is defined at coordinate $(i, j)$ as:

$$(I * K)_{i,j} = \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} I_{i+m, j+n} \cdot K_{m, n}$$

*Note: In deep learning frameworks, this operation is technically cross-correlation, as the kernel is typically not flipped before the dot product. However, since the kernel weights are learned, the distinction is mathematically negligible in this context.*

#### 3.2 The Sliding Window as a Linear Operator
The convolution operation can be viewed as a **sliding window** mechanism. For every valid position $(i, j)$ in the output grid, we extract a local patch of the input $P_{i,j} \in \mathbb{R}^{k \times k}$ and compute the Frobenius inner product with the kernel:

$$(I * K)_{i,j} = \langle P_{i,j}, K \rangle_F$$

This demonstrates that convolution is a linear operator that restricts the connectivity of the output. Instead of every output pixel being a function of every input pixel (as in a dense layer), each output pixel only 'sees' a small **local receptive field**.

#### 3.3 Parameter Sharing and Weight Tying
One of the most powerful aspects of convolution is **parameter sharing**. The same kernel $K$ is used to compute the output at every spatial location $(i, j)$. Mathematically, this means the weights of the linear transformation are 'tied' across the spatial dimensions, significantly reducing the number of degrees of freedom from $O(H^2 W^2)$ to $O(k^2)$.

## Toeplitz and Circulant Matrices

### Subtask:
Derive the Doubly Block Toeplitz matrix structure that represents a 2D convolution and prove that a convolutional layer is a restricted linear layer.


### 4. Toeplitz and Circulant Matrices

#### 4.1 From 1D Convolution to Toeplitz Matrices
In 1D, the discrete convolution of a signal $x$ with a kernel $k$ can be represented as a matrix-vector product $y = Ax$, where $A$ is a **Toeplitz matrix**. A Toeplitz matrix has the property that each descending diagonal from left to right is constant:
$$A = \begin{bmatrix} k_0 & 0 & 0 \\ k_1 & k_0 & 0 \\ k_2 & k_1 & k_0 \\ 0 & k_2 & k_1 \end{bmatrix}$$

#### 4.2 Doubly Block Toeplitz Matrices for 2D
To represent a 2D convolution as a single matrix multiplication, we first **vectorize** the input image $I \in \mathbb{R}^{H \times W}$ into a vector $\text{vec}(I) \in \mathbb{R}^{HW}$ (usually by concatenating rows).

The 2D convolution is then represented by a **Doubly Block Toeplitz (DBT)** matrix.
1. **Blocks**: The matrix is composed of blocks $B_i$, where each block is itself a Toeplitz matrix representing the convolution along one row.
2. **Block Toeplitz Structure**: The arrangement of these blocks follows a Toeplitz pattern, representing the shift across columns/rows.

$$M = \begin{bmatrix} B_0 & B_{-1} & B_{-2} \\ B_1 & B_0 & B_{-1} \\ B_2 & B_1 & B_0 \end{bmatrix}$$

#### 4.3 Convolution as a Restricted Linear Layer
A Fully Connected (FC) layer allows the matrix $M$ to be any arbitrary dense matrix in $\mathbb{R}^{OHW \times IHW}$. A Convolutional Layer is a **restricted linear layer** where $M$ is constrained to be:
- **Sparse**: Most entries are zero because the kernel has a small local support ($k \ll H, W$).
- **Structured**: The entries are shared (parameter sharing), resulting in the Doubly Block Toeplitz form.

This structure ensures that the linear operation is invariant to the absolute position of features, leading directly to the property of equivariance.

## Translation Equivariance Proof

### Subtask:
Provide a formal mathematical proof that convolution commutes with the shift operator and explain its significance for spatial data.


### 5. Translation Equivariance Proof

#### 5.1 Defining the Shift Operator
Let $I$ be an image in $\mathbb{R}^{H \times W}$. We define a 2D shift operator $S_{\Delta i, \Delta j}$ that translates the image by $(\Delta i, \Delta j)$:

$$(S_{\Delta i, \Delta j}I)_{i,j} = I_{i - \Delta i, j - \Delta j}$$

#### 5.2 Proof of Equivariance
We want to show that $S(I * K) = (SI) * K$.

Starting with the definition of convolution on a shifted image $(SI) * K$ at position $(i,j)$:
$$((SI) * K)_{i,j} = \sum_{m,n} (SI)_{i+m, j+n} K_{m,n}$$

Substitute the definition of the shift operator $(SI)_{x,y} = I_{x-\Delta i, y-\Delta j}$:
$$((SI) * K)_{i,j} = \sum_{m,n} I_{(i+m)-\Delta i, (j+n)-\Delta j} K_{m,n}$$

Rearranging the indices:
$$((SI) * K)_{i,j} = \sum_{m,n} I_{(i-\Delta i)+m, (j-\Delta j)+n} K_{m,n}$$

Notice that this summation is exactly the definition of the original convolution evaluated at the shifted coordinate $(i-\Delta i, j-\Delta j)$:
$$((SI) * K)_{i,j} = (I * K)_{i-\Delta i, j-\Delta j} = (S_{\Delta i, \Delta j}(I * K))_{i,j}$$

Therefore, **$S(I * K) = (SI) * K$**.

#### 5.3 Significance for Spatial Data
Translation equivariance implies that if an object in an image moves, its representation in the feature map moves by the same amount. This is the fundamental reason CNNs are effective for computer vision: the network does not need to re-learn what a 'cat' looks like at every possible pixel location. If a filter learns to detect a vertical edge, it can detect that edge anywhere in the image because the kernel weights are shared across all positions.

## Final Task

### Subtask:
Summarize how foundational linear algebra evolves into the specific tensor operations used in CNN architectures.


### 6. Final Task: From Linear Algebra to Convolutional Architectures

#### 6.1 The Mathematical Evolution
The derivation of CNNs is a journey of imposing **physical symmetries** onto **linear operators**. We began with the general concept of a vector space and linear transformations, which in their unconstrained form (Fully Connected layers), treat every input dimension as independent and every output as a global function of the input.

#### 6.2 Key Transitions
*   **From Vectors to Tensors**: By recognizing images as tensors $\mathbb{R}^{H \times W}$, we acknowledge the underlying spatial grid and the local topology of pixels.
*   **From Dense to Sparse**: By restricting the linear operator to a local receptive field, we transition from dense $O(N^2)$ matrices to sparse matrices, reflecting the physical reality that local correlations are most informative.
*   **From General Linear to Doubly Block Toeplitz**: By enforcing parameter sharing, we transform the sparse matrix into a structured **Doubly Block Toeplitz** form. This reduces the parameter space from millions of weights to a small $k \times k$ kernel.

#### 6.3 Geometric Convergence
The convergence of these properties results in **Translation Equivariance**. This mathematical property ensures that the network's internal representations are spatially consistent. The 'sliding window' is not just a heuristic; it is the unique linear operator (up to certain conditions) that commutes with the shift operator.

#### 6.4 Conclusion
Understanding CNNs through the lens of linear algebra and tensor calculus reveals that they are not 'black boxes' but are highly structured linear layers. This structure is precisely what allows deep learning models to generalize across high-dimensional spatial data efficiently and effectively.